## Stage 1: Data Cleaning and Preparation

##### GOAL: Clean and structure the dataset from the World Bank Group’s Climate Change Data to prepare it for predictive machine learning modeling of CO₂ emissions.”
  ##### Dataset Link: https://datacatalog.worldbank.org/dataset/climate-change-data

###  The dataset includes global data from 1990 to 2011 with features like:
1. CO2 emissions
2. Population data
3. GDP, GNI, FDI
4. Land usage
5. Climate & disasters

### Import all needed libraries:

1. Import & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import os

# Show all files in current directory
print("Files:", os.listdir())

# Set display to show all columns
pd.set_option('display.max_columns', None)

# Load dataset
df = pd.read_csv('climate_change_download_0.csv', encoding='utf-8', low_memory=False)

# Show shape and columns
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Show first 5 rows
df.head(10)



#### The complete Climate Change Dataset is imported into a pandas DataFrame from the downloaded file "climate_change_download_0.csv":

####  Global data overview:
##### A global overview of the imported data yields the following insights

In [ ]:
# 🌍 Global Data Overview

# 1. Dataset shape and data types
print(" Dataset Shape:", df.shape)
print("\n Column Data Types and Non-Null Counts:")
df.info()

# 2. Unique counts for key columns
print("\n Unique Indicators (Series):", df['Series name'].nunique())
print(" Unique Countries:", df['Country name'].nunique())

# 3. Check Year Columns (starting from index 10)
print("\n Detected Year Columns:")
print(df.columns[20:].tolist())  # Adjust index if needed



##### Initial Project Goals
##### Initial Goals:

1. Remove unnecessary metadata
2. Convert series into column-based structure
3. Prepare country-year-feature format
4. Handle missing/null values
5. Prepare a machine learning-ready flat dataset

### Data Cleaning

In [ ]:
# Remove rows marked as "Text" in "SCALE" and "Decimals"


df = df[(df['SCALE'] != 'Text') & (df['Decimals'] != 'Text')]

In [ ]:
import numpy as np

# List of metadata columns to drop
drop_cols = ['Country name', 'Series code', 'SCALE', 'Decimals']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# Replace ".." and empty strings with NaN
df.replace({'..': np.nan, '': np.nan}, inplace=True)

# Convert year columns to numeric
year_columns = df.columns[2:]  # Assuming first two columns are identifiers
df[year_columns] = df[year_columns].apply(pd.to_numeric, errors='coerce')

# Clean and standardize Series names
df['Series name'] = df['Series name'].str.strip().str.replace(' ', '_', regex=True).str.lower()


### DataFrame Transformation:


In [ ]:
#  STEP 1: Melt the DataFrame
# Convert from wide format (years as columns) to long format
df_melted = df.melt(
    id_vars=['Country code', 'Series name'],
    var_name='Year',
    value_name='Value'
)

#  Drop rows with missing values in 'Value'
df_melted.dropna(subset=['Value'], inplace=True)

#  Optional: Ensure 'Year' is treated as integer (not string)
df_melted['Year'] = pd.to_numeric(df_melted['Year'], errors='coerce')


In [ ]:

#  STEP 2: Pivot the DataFrame
# Turn 'Series name' values into column headers
df_pivot = df_melted.pivot_table(
    index=['Country code', 'Year'],
    columns='Series name',
    values='Value'
).reset_index()


In [ ]:
#  STEP 3: Cleanup
# Remove column name level caused by pivot
df_pivot.columns.name = None

#  Preview the transformed DataFrame
df_pivot.head()




### Handle Missing Values:


 ##### The goal is to minimize missing data while retaining as much useful information as possible. Instead of dropping all rows with NaNs, the approach filters by year, country, and feature—removing rows or columns starting with those that have the most missing values.Since countries and years appear multiple times, NaN counts are aggregated per unique country and year to guide selective cleaning.

##### Filtering the years by missing values: 
Checking the amount of missing values for each year:

In [ ]:
# Detect % of missing values


missing_percent = df_pivot.isnull().mean().sort_values(ascending=False) * 100
missing_percent[missing_percent > 0]  # View only columns with missing values

    

#### Filtering the countries by missing values
The same procedure is applied to the filtering of countries with missing values. The following snippet shows the number of NaNs for each country.

In [ ]:
# Drop columns with >50% missing values (adjust as needed)


df_pivot = df_pivot.loc[:, df_pivot.isnull().mean() < 0.5]
# Drop rows with any missing values


df_clean = df_pivot.dropna()

Export Clean Dataset:

In [ ]:
df_clean.to_csv("cleaned_climate_data.csv", index=False)
print("Exported cleaned data to climate_clean_data.csv ✅")


  ##### Now that the dataset has been rearranged and cleaned of missing values, it can be exported to a csv file (without the row index) for further analysis:

 ### Summary:
1. Cleaned dataset has shape: (rows, columns)
2. Removed text rows, unneeded columns
3. Converted ".." and empty cells to NaN
4. Converted wide to long format and pivoted into machine learning-friendly format
5. Dropped excessive missing data
6. Exported cleaned CSV